# BinaryMatchboxNet KWS — 작업 노트북

**규칙 하나**: 셀 1을 돌린 뒤에는 **어느 셀이든 단독으로** 실행할 수 있다.
셀 사이에 변수를 넘기지 않는다 — 분석 셀은 전부 체크포인트에서 시작한다.

이 규칙이 없어서 겪은 일: 커널 stale로 100 에폭 ×2 낭비, `set_seed` 미정의,
`t` 미정의. 전부 셀 간 암묵적 의존 때문이었다.

> **커널을 재시작했으면 반드시 셀 1부터.** `git pull`은 프로젝트 모듈을
> import 하기 *전에* 일어나야 하고, 이미 import 된 모듈은 pull 해도 안 바뀐다.

### 순차 실행해도 안전하다

**학습을 시작하는 셀은 하나도 없다.** §9의 `train(...)`은 전부 주석이고,
돌리고 싶은 줄 하나만 직접 풀어야 한다 — 100 에폭짜리가 실수로 시작되면
안 되기 때문이다.

실제로 일을 하는 셀은 넷뿐이고 나머지는 함수 정의다:

| 셀 | 하는 일 | 걸리는 시간 |
|---|---|---|
| §1 프리플라이트 | pull + stale 검사 | 초 |
| 런 목록 | `runs/`를 훑어 태그·정규화를 표로 | 초 |
| §5 창 위치 곡선 | 위치 9개 × 채움 2종 = 18회 평가 | 수 분 |
| §7 레벨 보정 | 엔벨로프 통계 1회 | 1~2분 |

§8 frac 스윕은 **`xlse` 런 태그를 넣어야** 돌아간다. 태그는 런 목록에서 고른다.

## 1. 프리플라이트 — 커널 재시작 후 항상 여기부터

In [ ]:
import os, sys, subprocess, pathlib
os.chdir(os.path.expanduser('~/KWS-AFE-Digital')); sys.path.insert(0, os.getcwd())

print(subprocess.run(['git','pull'], capture_output=True, text=True).stdout.strip())
print(subprocess.run(['git','log','--oneline','-1'], capture_output=True,
                     text=True).stdout.strip(), '\n')

# 파일이 아니라 '로드된 모듈'을 검사한다. 파일만 보면 stale 커널을 못 잡는다.
import torch, numpy as np
from train.config import load_config, AFEConfig
from data.speech_commands import build_dataloaders
from data.afe import AFEFrontend
from models.binary_matchboxnet import BinaryMatchboxNet
from train.train import Trainer, set_seed
import data.afe as _A
import experiments.window_offset as _W
import experiments.level_calibration as _L

NEED = [('spice 경로 analog/ 이전', ('train/config.py', 'analog/AFE/artifacts'),
         lambda: AFEConfig().spice_matrix_path.startswith('analog/')),
        ("normalize='xmix'", ('data/afe.py', 'def _xmix'),
         lambda: hasattr(_A.AFEFrontend, '_xmix')),
        ("normalize='xlse'", ('data/afe.py', 'def _xlse'),
         lambda: hasattr(_A.AFEFrontend, '_xlse')),
        ('effective_alpha()', ('data/afe.py', 'def effective_alpha'),
         lambda: hasattr(_A.AFEFrontend, 'effective_alpha')),
        ('비교기 k개', ('train/config.py', 'comparators_per_channel'),
         lambda: 'comparators_per_channel' in AFEConfig.__dataclass_fields__),
        ('창 위치 곡선', ('experiments/window_offset.py', 'def offset_curve'),
         lambda: hasattr(_W, 'offset_curve')),
        ('레벨 보정', ('experiments/level_calibration.py', 'def level_stats'),
         lambda: hasattr(_L, 'level_stats'))]
bad_disk, bad_mem = [], []
for name, (f, needle), check in NEED:
    on_disk = needle in pathlib.Path(f).read_text()
    in_mem = bool(check())
    print(f"{'✅' if in_mem else '❌'} {name:<24} 파일 {'O' if on_disk else 'X'}"
          f"  모듈 {'O' if in_mem else 'X'}")
    (bad_disk if not on_disk else bad_mem if not in_mem else []).append(name)
if bad_disk:
    raise RuntimeError(f'파일에 없음 → git pull 실패: {bad_disk}')
if bad_mem:
    raise RuntimeError(f'파일엔 있는데 모듈엔 없음 → **커널 재시작**: {bad_mem}')

DATA_ROOT = os.path.expanduser('~/datasets/speech_commands_v2')
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000
assert os.path.isdir(DATA_ROOT), DATA_ROOT
print(f'\n✅ 준비 완료   {DEV}   {DATA_ROOT}')

## 2. 확정 설정

여기 한 곳만 고치면 아래 전부가 따라간다. 확정 근거는 `proposal/`.

| | 값 | 근거 |
|---|---|---|
| 정규화 | `xmix`, floor 0.02 | 저항 분압으로 구현 가능, minmax 대비 −0.35pp |
| 비교기 | 채널당 1개 | 2개면 +2.5pp (업그레이드 경로) |
| 필터뱅크 | `spice` 16채널 | 동료 v15 보드와 16/16 일치 |
| 대역 | 50–8000 Hz | 125–5000은 −1.6pp |

> **`xmix` vs `xlse`**: `xmix`는 다이오드-OR이 **진짜 max**를 준다고 본다.
> 실제 다이오드는 지수 함수라 소프트 max(`xlse`)를 준다. `xmix` 런의 숫자는
> *측정으로서는 정확하되*, **평범한 다이오드-OR로는 만들 수 없는 프론트엔드**의
> 값이다 — 상한으로 읽어야 한다. `xlse`는 §6-6, `docs/diagrams/14_diode_softmax.svg`.

In [ ]:
BASE = {'afe.filterbank_source': 'spice',
        'afe.compression': 'sqrt',
        'afe.normalize': 'xmix',
        'afe.xmax_floor_frac': 0.02,
        'afe.comparators_per_channel': 1,
        'model.in_channels': 16}

def make_cfg(tag, **over):
    """확정 설정 + 이번 실험만의 override."""
    cfg = load_config('configs/base.yaml', {'tag': tag, **BASE, **over})
    cfg.data.root = DATA_ROOT
    return cfg

print(make_cfg('probe').afe)

## 3. 학습

`d > 0`과 α 범위를 **학습 시작 전에** 검사한다. `d = 0`이면 floor가 반영되지
않은 것이고, 그대로 두면 100 에폭을 버린 뒤에야 알게 된다.

In [ ]:
def train(tag, **over):
    cfg = make_cfg(tag, **over)
    set_seed(cfg.train.seed)
    afe = AFEFrontend(cfg.afe); model = BinaryMatchboxNet(cfg.model)
    tr, va, te = build_dataloaders(cfg.data, cfg.train.batch_size, SR,
                                   seed=cfg.train.seed)
    w = next(iter(tr))[0]
    afe.init_fixed_scale(w)      # δ (그리고 xlse면 lse_temp) 먼저
    afe.init_thresholds(w)       # 그 다음 α

    k = cfg.afe.comparators_per_channel
    d, a = float(afe.xmax_floor), afe.effective_alpha()
    assert d > 0, f'd=0 — floor_frac({cfg.afe.xmax_floor_frac})이 반영 안 됨'
    n_par = sum(p.numel() for p in model.parameters())
    extra = ''
    if cfg.afe.normalize == 'xlse':
        extra = (f"  T={float(afe.lse_temp):.5f} "
                 f"(frac {cfg.afe.lse_temp_frac})")
    print(f'[{tag}] {cfg.afe.normalize}  d={d:.5f}{extra}  '
          f'입력 {cfg.afe.n_channels*k}행  파라미터 {n_par:,}  '
          f'α init {a.min():.3f}~{a.max():.3f}')

    t = Trainer(cfg, model, afe=afe)
    t.fit(tr, va, resume=True)
    return report(tag)

print('train(tag, **override) 준비됨')

## 4. 평가 — 체크포인트에서만 시작한다

학습 셀을 안 돌렸어도, 커널을 재시작했어도 동작한다.

> **설정은 `runs/<tag>/config.yaml`에서 읽는다** — `BASE`가 아니라.
> 런마다 `normalize`가 다를 수 있고(`xmix` / `xlse`), 틀린 정규화로 평가해도
> **에러 없이 그럴듯한 숫자**가 나온다.

In [ ]:
def load_run(tag, **over):
    """그 런이 실제로 학습한 config로 복원한다. 이전 셀에 의존하지 않는다."""
    saved = f'runs/{tag}/config.yaml'
    cfg = (load_config(saved, dict(over)) if os.path.isfile(saved)
           else make_cfg(tag, **over))
    cfg.data.root = DATA_ROOT
    afe = AFEFrontend(cfg.afe).to(DEV).eval()
    model = BinaryMatchboxNet(cfg.model).to(DEV).eval()
    ck = torch.load(f'runs/{tag}/best.pt', map_location=DEV, weights_only=True)
    model.load_state_dict(ck['model']); afe.load_state_dict(ck['afe'])
    return cfg, afe, model, ck

def test_loader(cfg):
    return build_dataloaders(cfg.data, cfg.train.batch_size, SR,
                             seed=cfg.train.seed)[2]

@torch.no_grad()
def accuracy(afe, model, loader, T, shift_ms=0.0, gain_db=0.0):
    """shift_ms > 0 = 소리가 늦게 들어옴 (밖으로 나간 부분은 버려진다)."""
    k = int(round(shift_ms * SR / 1000)); ok = n = 0
    g = 10.0 ** (gain_db / 20.0)
    for x, y in loader:
        x, y = x.to(DEV), y.to(DEV)
        if k:
            x = torch.roll(x, k, dims=-1)
            if k > 0: x[..., :k] = 0.0
            else:     x[..., k:] = 0.0
        if gain_db: x = x * g
        ok += (model(afe(x, target_T=T)).argmax(1) == y).sum().item()
        n += y.numel()
    return ok / n

def report(tag, **over):
    cfg, afe, model, ck = load_run(tag, **over)
    a = afe.effective_alpha()
    dead = int(((a >= 0.99) | (a <= 0.01)).sum())
    acc = accuracy(afe, model, test_loader(cfg), cfg.model.T)
    print(f'>>> {tag}  [{cfg.afe.normalize}]  val {ck["best_acc"]:.4f}   '
          f'test {acc:.4f}')
    print(f'    d={float(afe.xmax_floor):.5f}  α {a.min():.3f}~{a.max():.3f}  '
          f'죽은 비교기 {dead}/{a.numel()}')
    return acc

print('load_run / test_loader / accuracy / report 준비됨')

### 런 목록 — 어떤 태그가 있고 무엇으로 학습됐나

**아래 분석 셀에 넣을 태그를 여기서 고른다.** `normalize`가 `xmix`인지
`xlse`인지가 특히 중요하다 — §8 frac 스윕은 `xlse` 런에만 의미가 있다.

In [ ]:
import glob, yaml

def list_runs():
    rows = []
    for p in sorted(glob.glob('runs/*/config.yaml')):
        tag = os.path.basename(os.path.dirname(p))
        r = yaml.safe_load(open(p)) or {}
        a, d = r.get('afe') or {}, r.get('data') or {}
        ck = f'runs/{tag}/best.pt'
        best = (torch.load(ck, map_location='cpu', weights_only=True)
                .get('best_acc') if os.path.isfile(ck) else None)
        rows.append((tag, a.get('normalize'), a.get('lse_temp_frac'),
                     a.get('comparators_per_channel'),
                     d.get('aug_time_shift_ms'), d.get('aug_gain_db'), best))
    if not rows:
        print('runs/ 가 비어 있다 — 아직 학습한 런이 없다'); return []
    print(f"{'태그':<18}{'정규화':>8}{'frac':>7}{'k':>3}"
          f"{'shift':>7}{'gain':>12}{'val':>8}")
    for tag, nz, fr, k, sh, gn, b in rows:
        print(f'{tag:<18}{str(nz):>8}{(fr if nz=="xlse" else "-"):>7}'
              f'{str(k):>3}{str(sh or 0):>7}{str(gn or "-"):>12}'
              f'{(f"{b:.4f}" if b else "-"):>8}')
    return [r[0] for r in rows]

list_runs()

## 5. 창 위치 곡선 — 슬라이딩의 **진짜** 비용

FPGA는 100 ms마다 판정하는 슬라이딩 창을 쓴다. 한 단어가 창 8개에 걸리고
창마다 위치가 다르므로, 그 위치 변화를 견뎌야 한다.

단어를 **자르지 않고** 창 안에서만 옮기고, 나머지는 그 클립 자신의 노이즈
플로어로 채운다. 위치는 정규화 p (0 = 창 왼쪽 밀착, 1 = 오른쪽 밀착).

**읽는 법 — `noise` 열의 낙폭:**

| 낙폭 | 판단 |
|---|---|
| ≤ 5pp | 슬라이딩이 사실상 공짜. 최대 확신도 선택만 붙이면 끝 |
| 5–15pp | 최대 확신도 선택 필요 |
| ≥ 15pp | 오정렬이 진짜 문제 — **긴 캔버스** 증강을 만들 가치가 있음 |

`zero` 열이 크게 낮으면 인공 무음에서 상대 임계가 퇴화한 것(§4-4).

In [ ]:
from experiments.window_offset import offset_curve, print_offset_curve

def offset_report(tag, steps=9, fills=('noise','zero'), **over):
    cfg, afe, model, _ = load_run(tag, **over)
    res = offset_curve(afe, model, test_loader(cfg), cfg.model.T,
                       steps=steps, fills=fills, device=DEV)
    print_offset_curve(res, tag)
    return res

TAG = 'af_k1_ref'          # ← 위 런 목록에서 고른다
offset_report(TAG)

## 6. 시간 이동 곡선 — ⚠️ 잘림을 잰다

1초 클립을 그냥 밀기 때문에 단어가 클립 밖으로 나가 **사라진다**. 그래서 이
곡선의 −11pp는 오정렬이 아니라 대부분 **잘림**이고, 증강으로 못 고친다
(±300 증강 실측 +0.33pp).

**슬라이딩 설계 판단에는 §5를 쓴다.** 이 셀은 비교·기록용으로만 남긴다.
그림: `docs/diagrams/13_sliding_window.svg`.

In [ ]:
def shift_curve(tag, shifts=(-400,-300,-200,-100,0,100,200,300,400), **over):
    cfg, afe, model, _ = load_run(tag, **over)
    te = test_loader(cfg)
    base = accuracy(afe, model, te, cfg.model.T, 0.0)
    print(f'{tag}\n{"이동(ms)":>9}{"test acc":>10}{"기준 대비":>11}')
    out = {}
    for s in shifts:
        a = accuracy(afe, model, te, cfg.model.T, s)
        out[s] = a
        print(f'{s:>9}{a:>10.4f}{(a-base)*100:>+10.1f}pp')
    return out

print('shift_curve(tag) 준비됨 — 판단은 §5로')

## 7. 레벨 보정 — GSC가 배치 음량 범위를 덮는가

`xlse`의 `lse_temp_frac`은 고르는 값이 아니라 **1/음량**이다
(frac = n·V_T / 엔벨로프 스윙, 분자는 물리가 26–39 mV로 고정).
프리앰프는 이미 ×10 상한이라 범위를 좁힐 수단이 없다:
**60~85 dB SPL = 25 dB = frac 0.55~0.03.**

GSC는 자원자마다 다른 폰·거리로 녹음됐고 로더가 레벨을 정규화하지 않으므로,
**우연히** 그 범위를 이미 줄 수도 있다. 그걸 잰다.

| 결과 | 다음 |
|---|---|
| GSC 폭 ≥ 25 dB | 학습이 이미 덮음 — 게인 증강 불필요 |
| GSC 폭 < 25 dB | 모자란 만큼 `aug_gain_db`를 셀이 계산해준다 |

§0의 앵커 점검도 같이 본다 — `init_fixed_scale`이 **무음 프레임까지 섞어**
중앙값을 내므로 앵커가 아래로 끌려갔을 수 있다.

In [ ]:
from experiments.level_calibration import level_stats, print_level_report

def level_report(tag, split='test', spl=74.0, **over):
    cfg, afe, _, _ = load_run(tag, **over)
    tr, _, te = build_dataloaders(cfg.data, cfg.train.batch_size, SR,
                                  seed=cfg.train.seed)
    st = level_stats(afe, tr if split == 'train' else te,
                     cfg.afe.compression, DEV)
    print_level_report(st, float(cfg.afe.lse_temp_frac), spl,
                       f'{tag} [{split}]')
    return st

TAG = 'af_k1_ref'          # ← 위 런 목록에서 고른다
level_report(TAG, split='train')

## 8. frac 스윕 — 앵커가 얼마나 중요한가 (재학습 없음)

`lse_temp`는 **버퍼**라 테스트 시점에 바꿀 수 있다. 즉 "진짜 회로가 학습
가정보다 소프트/하드하면 얼마나 잃나"를 **100 에폭 없이** 바로 잰다.

frac ∝ 1/음량이므로 배율 m은 곧 **−20·log₁₀(m) dB SPL**이다.
m=2 → 6 dB 더 조용한 말, m=0.5 → 6 dB 더 큰 말.

**이게 "이전 정확도가 틀렸나"에 대한 답이다:**
곡선이 평평하면 앵커를 어디에 잡았든 상관없고 숫자는 그대로 유효하다.
무너지면 그 숫자는 **한 가지 음량에서만 성립하는 값**이다.

`xlse` 런에만 의미가 있다 (`xmix`에는 `lse_temp`가 없다).

In [ ]:
import math

@torch.no_grad()
def frac_sweep(tag, mults=(0.25, 0.5, 1.0, 2.0, 4.0, 8.0), **over):
    cfg, afe, model, _ = load_run(tag, **over)
    if cfg.afe.normalize != 'xlse':
        print(f'{tag}: normalize={cfg.afe.normalize} — lse_temp 없음. '
              f'xlse 런에만 쓴다.')
        return None
    te = test_loader(cfg)
    t0, f0 = float(afe.lse_temp), float(cfg.afe.lse_temp_frac)
    # 학습점(m=1)을 먼저 잰다 — 루프 안에서 잡으면 m<1 행에 기준이 없다.
    base = accuracy(afe, model, te, cfg.model.T)
    print(f'{tag}   학습 frac {f0}   T={t0:.5f}   학습점 {base:.4f}\n')
    print(f"{'배율':>6}{'frac':>8}{'≈음량':>9}{'test':>9}{'학습점 대비':>12}")
    out = {}
    for m in mults:
        afe.lse_temp.fill_(t0 * m)
        a = accuracy(afe, model, te, cfg.model.T)
        out[m] = a
        print(f'{m:>6.2f}{f0*m:>8.3f}{-20*math.log10(m):>+8.0f}dB'
              f'{a:>9.4f}{(a-base)*100:>+11.1f}pp')
    afe.lse_temp.fill_(t0)
    v = list(out.values())
    print(f'\n낙폭 {(max(v)-min(v))*100:.1f}pp  '
          f'→ ' + ('앵커 무관, 숫자 유효 ✅' if (max(v)-min(v)) < 0.05
                  else '앵커에 민감 — 게인 증강 필요 ⚠️'))
    return out

print('frac_sweep(tag) 준비됨')

# 위 '런 목록' 셀에서 normalize=xlse 인 태그를 골라 넣는다.
# frac_sweep('xl_ref')

## 9. 학습 방법 탐색

각 런 100 에폭. **하나씩 돌리고 결과를 보고 다음을 정한다.**

| 실험 | 무엇에 답하나 | 상태 |
|---|---|---|
| A. time-shift 증강 | 슬라이딩 오정렬 | ❌ **끝남** — +0.33pp, 잘림이라 못 고침 |
| B. 게인 증강 (`xlse`) | 음량 범위 | ⬅️ **지금 할 것** |
| C. `spice_gain_restore` | 소프트 max의 채널간 스케일 | ⬅️ 같이 |

**B가 왜 다시 살아났나**: 게인 증강은 예전에 −9pp로 기각됐지만 그건
`normalize=fixed` 얘기였다 (고정 임계 아래서 게인은 이미지를 지운다).
`xlse`에서는 클립 게인을 바꾸는 것이 **`lse_temp_frac`을 바꾸는 것과 같다** —
교란이 아니라 우리가 강건해지고 싶은 바로 그 물리량이다.

**C가 왜 필요한가**: `spice_gain_restore=False`의 근거는 "채널별 threshold가
1.8 dB 편차를 흡수한다"였다. **진짜 max에서는 맞지만 소프트 max에서는 틀리다** —
LSE 분모는 16채널을 한 숫자로 섞으므로 한 채널의 오차가 **모두의** 분모를 민다.

> 게인 값은 §7이 계산해준 값을 쓰는 게 낫다. 아래 6/12 dB는 자리표시자.

In [ ]:
XLSE = {'afe.normalize': 'xlse', 'afe.lse_temp_frac': 0.13}

# ⚠️ 전부 주석이다. 순차 실행이 100 에폭을 시작하면 안 된다.
#    돌릴 줄 하나만 풀고, 끝나면 다시 주석 처리한다.

# 0) 기준선 — xlse에서 증강 없이. 나머지는 전부 이것과 비교한다.
# train('xl_ref', **XLSE)

# B) 게인 증강. §7이 계산해준 폭을 넣는다.
# for g in (6, 12):
#     train(f'xl_gain{g}', **XLSE,
#           **{'data.aug_gain_db': [-float(g), float(g)]})

# C) 채널간 스케일 복원 (소프트 max에서만 의미가 생긴다)
# train('xl_gainrestore', **XLSE, **{'afe.spice_gain_restore': True})

# 끝나면 비교
# for tag in ('xl_ref', 'xl_gain6', 'xl_gain12', 'xl_gainrestore'):
#     frac_sweep(tag); print()

## 10. 하드웨어 내보내기 — α → 저항비

**반드시 `effective_alpha()`로 읽는다.** `xmix`/`xlse`는 α를 forward에서
straight-through로 클램프하므로 파라미터 원값은 [0,1] 밖으로 떠다닌다.

In [ ]:
def alpha_table(tag='af_k1_ref', RTOT=1e6, **over):
    cfg, afe, _, _ = load_run(tag, **over)
    a = afe.effective_alpha()
    fc = [166,295,447,631,832,1072,1349,1660,
          2042,2455,2951,3467,4169,4898,5754,6761]
    k = cfg.afe.comparators_per_channel
    print(f'{tag} [{cfg.afe.normalize}]  δ={float(afe.xmax_floor):.5f}  '
          f'Ra+Rb={RTOT/1e3:.0f} kΩ\n')
    print(f"{'ch':>2}{'f_c[Hz]':>9}{'i':>3}{'α':>8}{'Rb[kΩ]':>9}{'Ra[kΩ]':>9}{'상태':>7}")
    for c in range(cfg.afe.n_channels):
        for i in range(k):
            x = float(a[c*k + i]); rb = RTOT*x
            st = '죽음' if x >= 0.99 or x <= 0.01 else '정상'
            print(f'{c:>2}{fc[c]:>9}{i:>3}{x:>8.4f}{rb/1e3:>9.1f}'
                  f'{(RTOT-rb)/1e3:>9.1f}{st:>7}')
    ok = bool(a.min() >= 0 and a.max() <= 1)
    print(f"\n범위 {a.min():.3f}~{a.max():.3f}  "
          f"{'제작 가능 ✅' if ok else '제작 불가 ⚠️'}")

TAG = 'af_k1_ref'          # ← 내보낼 런
alpha_table(TAG)

## 11. 잘못된 런 지우기

`resume=True`라 같은 태그로 다시 돌리면 **이어서** 학습한다. 설정을 바꿨으면
반드시 지우고 시작해야 한다.

In [ ]:
import shutil
for tag in []:                      # 예: ['xl_ref']
    p = f'runs/{tag}'
    shutil.rmtree(p, ignore_errors=True); print('삭제:', p)